Incorporating the δ term to the instrumental-variable moment expressions

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import GMM

# Load the data using a simple raw text path
file_path = 'midterm_partone.csv'
input_table = pd.read_csv(file_path)

# First-stage regression (OLS)
model_iv = sm.OLS(input_table["Inventory Turnover"], 
                  input_table[["Constant", "Current Ratio", "Quick Ratio", "Debt Asset Ratio"]]).fit()
endog_predict = model_iv.predict(input_table[["Constant", "Current Ratio", "Quick Ratio", "Debt Asset Ratio"]])
input_table["Endogenous Param"] = endog_predict

# Second-stage regression (OLS)
model_2sls = sm.OLS(input_table["Stock Change"], 
                    input_table[["Constant", "Endogenous Param", "Operating Profit", "Interaction Effect"]]).fit()
model_2sls.summary()

# Set up data for GMM
y_vals = np.array(input_table["Stock Change"])
x_vals = np.array(input_table[["Inventory Turnover", "Operating Profit", "Interaction Effect"]])
iv_vals = np.array(input_table[["Current Ratio", "Quick Ratio", "Debt Asset Ratio"]])

# Define a custom GMM model incorporating delta
class gmm_with_delta(GMM):
    def __init__(self, *args, delta=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.delta = delta
    
    def momcond(self, params):
        p0, p1, p2, p3 = params
        endog = self.endog
        exog = self.exog
        inst = self.instrument

        # Errors from regression equation
        error = endog - p0 - p1 * exog[:, 0] - p2 * exog[:, 1] - p3 * exog[:, 2]

        # Moment conditions with the delta term included
        error0 = error + self.delta * 1
        error1 = (error + self.delta * 1) * exog[:, 1]
        error2 = (error + self.delta * 1) * exog[:, 2]
        error3 = (error + self.delta * 1) * inst[:, 0]
        error4 = (error + self.delta * 1) * inst[:, 1]
        error5 = (error + self.delta * 1) * inst[:, 2]

        g = np.column_stack((error0, error1, error2, error3, error4, error5))
        return g

# Initial parameter values for GMM
beta0 = np.array([0.1, 0.1, 0.1, 0.1])

# Fit the GMM model incorporating delta
# Assuming a non-zero value for delta (as per the expert's claim)
delta = 0.05
res = gmm_with_delta(endog=y_vals, exog=x_vals, instrument=iv_vals, k_moms=6, k_params=4, delta=delta).fit(beta0)

# Print the summary of the GMM results
print(res.summary())

Optimization terminated successfully.
         Current function value: 0.000046
         Iterations: 9
         Function evaluations: 13
         Gradient evaluations: 13
Optimization terminated successfully.
         Current function value: 0.000373
         Iterations: 7
         Function evaluations: 13
         Gradient evaluations: 13
Optimization terminated successfully.
         Current function value: 0.000372
         Iterations: 6
         Function evaluations: 10
         Gradient evaluations: 10
Optimization terminated successfully.
         Current function value: 0.000372
         Iterations: 1
         Function evaluations: 3
         Gradient evaluations: 3
                            gmm_with_delta Results                            
Dep. Variable:                      y   Hansen J:                       0.6317
Model:                 gmm_with_delta   Prob (Hansen J):                 0.729
Method:                           GMM                                         
Da